# SpecStory Prompt Miner + Representation Learning (Colab / GPU)

This notebook does two things end to end:

1. **Mine + extract** real developer prompts from public GitHub repos that contain SpecStory session logs (`.specstory/history/*.md`).
2. **Learn representations** of those prompts on GPU: sentence embeddings, clustering, and 2D visualization.

The mining/extraction steps are network-bound (GitHub API). The **GPU is used in the embedding and clustering steps**, which is the machine-learning core of the project.

**Before running:** set `Runtime -> Change runtime type -> GPU`.

**Incremental mining:** Step 7 uses a **cache** in `outputs/cache/` and targets **500–800 repos that actually contain extractable SpecStory prompts** (not just discovered candidates). Each run stops after a time cap — re-run to keep growing the cache.

**You can either:**
- run the incremental miner (Step 7) to build/extend a SpecStory dataset, **or**
- skip to *Step 8b* and upload a `specstory_prompts_*.csv` you already produced locally.

## Step 1 — Install dependencies

In [ ]:
!pip -q install sentence-transformers umap-learn scikit-learn pandas requests matplotlib

from pathlib import Path
import subprocess
import sys

if not Path("src/github_repo_miner/data_cleaning.py").exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/medhab123/Prompt_Research.git", "_prompt_research_repo"],
        check=True,
    )
    sys.path.insert(0, str(Path("_prompt_research_repo/src").resolve()))
elif str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

## Step 2 — Confirm the GPU is available

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("CUDA available. GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("No GPU detected. Set Runtime -> Change runtime type -> GPU, then re-run.")

## Step 3 — GitHub tokens + config (5-token pool)

More tokens = more requests before you hit a rate limit. The pool **rotates to the next token automatically** when one is throttled.

Provide tokens via Colab secrets (key icon in sidebar) or environment variables:
- **`gh1`, `gh2`, `gh3`, `gh4`, `gh5`** (recommended)
- Also accepts `GH1`…`GH5`, `GITHUB_TOKEN`, `GH_TOKEN`

In [ ]:
import os, re, time, base64
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from dataclasses import dataclass
from getpass import getpass

import requests
import pandas as pd

# Names checked in environment variables and Colab secrets (gh1-gh5 primary).
TOKEN_ENV_VARS = [
    "gh1", "gh2", "gh3", "gh4", "gh5",
    "GH1", "GH2", "GH3", "GH4", "GH5",
    "GITHUB_TOKEN", "GH_TOKEN",
]


def _collect_tokens():
    found = []
    for var in TOKEN_ENV_VARS:
        val = os.getenv(var)
        if val:
            found.append(val.strip())
    try:
        from google.colab import userdata
        for var in TOKEN_ENV_VARS:
            try:
                val = userdata.get(var)
                if val:
                    found.append(val.strip())
            except Exception:
                pass
    except Exception:
        pass
    seen, unique = set(), []
    for tok in found:
        if tok and tok not in seen:
            seen.add(tok)
            unique.append(tok)
    return unique


_tokens = _collect_tokens()
if not _tokens:
    raw = getpass("Paste up to 5 GitHub tokens, comma-separated (blank = anon): ").strip()
    _tokens = [t.strip() for t in raw.split(",") if t.strip()]


class TokenPool:
    """Round-robin pool of GitHub tokens with rate-limit aware rotation."""

    def __init__(self, tokens):
        self.tokens = list(tokens) if tokens else [None]
        self.idx = 0

    @property
    def has_token(self):
        return any(t for t in self.tokens)

    def __len__(self):
        return len(self.tokens)

    def headers(self):
        token = self.tokens[self.idx]
        h = {"Accept": "application/vnd.github+json"}
        if token:
            h["Authorization"] = f"Bearer {token}"
        return h

    def rotate(self):
        self.idx = (self.idx + 1) % len(self.tokens)
        return self.tokens[self.idx]


TOKEN_POOL = TokenPool(_tokens)
BASE_URL = "https://api.github.com"

if TOKEN_POOL.has_token:
    n_real = len([t for t in TOKEN_POOL.tokens if t])
    print(f"Loaded {n_real} GitHub token(s). Code search enabled; "
          f"effective rate limit scales with token count.")
else:
    print("No tokens: repo search only, low rate limits, code search disabled.")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## Step 4 — SpecStory discovery queries

This notebook mines **only** SpecStory session logs (`.specstory/history/*.md`) — real developer prompts from Cursor/agent chat transcripts.

| Signal | GitHub filter | What you get |
|--------|---------------|--------------|
| README mention | `specstory in:readme` | Repos that document SpecStory usage |
| Topic tag | `topic:specstory` | Repos tagged with SpecStory |
| Path hint | `path:.specstory` | Repos containing SpecStory files |
| History path | `path:.specstory/history` | Exact session log directory |
| SpecStory root | `path:.specstory` | Any file under `.specstory` |
| Markdown logs | `extension:md path:.specstory` | All SpecStory markdown files |

Code-search queries require at least one GitHub token. No star/age filters are applied — we maximize coverage.

In [ ]:
# SpecStory-only discovery queries (mirrors src/github_repo_miner/queries.py)
SPECSTORY_REPO_QUERIES = [
    ("specstory_readme", "specstory in:readme"),
    ("topic_specstory", "topic:specstory"),
    ("specstory_path", "path:.specstory"),
]

SPECSTORY_CODE_QUERIES = [
    ("specstory_history", "path:.specstory/history"),
    ("specstory_root", "path:.specstory"),
    ("specstory_md", "extension:md path:.specstory"),
]

REPO_DETECTION_QUERIES = list(SPECSTORY_REPO_QUERIES)
CODE_SEARCH_QUERIES = list(SPECSTORY_CODE_QUERIES)

print(f"Repo-search queries: {len(REPO_DETECTION_QUERIES)} | "
      f"Code-search queries: {len(CODE_SEARCH_QUERIES)}")

## Step 5 — GitHub API helpers (search + pagination)

In [ ]:
GITHUB_SEARCH_RESULT_CAP = 1000


def _gh_get(url, params=None, timeout=30):
    """GET with multi-token rotation.

    Rotates to the next token on a rate-limit response and only sleeps once
    every token in the pool has been tried. Returns the final Response object
    (callers check status_code) or None on repeated network errors.
    """
    n = len(TOKEN_POOL)
    rate_hits = 0
    net_errors = 0
    while True:
        try:
            response = requests.get(url, headers=TOKEN_POOL.headers(), params=params, timeout=timeout)
        except requests.RequestException:
            net_errors += 1
            if net_errors >= 3:
                return None
            time.sleep(3)
            continue

        if response.status_code in (403, 429):
            rate_hits += 1
            if rate_hits < n:
                TOKEN_POOL.rotate()
                continue
            reset_ts = int(response.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait = min(max(reset_ts - time.time() + 2, 15), 120)
            print(f"   All {n} token(s) limited. Waiting {int(wait)}s...", flush=True)
            time.sleep(wait)
            rate_hits = 0
            TOKEN_POOL.rotate()
            continue

        return response


def _search_page(endpoint, query, *, page, per_page):
    params = {"q": query, "sort": "updated", "order": "desc", "per_page": per_page, "page": page}
    response = _gh_get(f"{BASE_URL}/{endpoint}", params=params)
    if response is None:
        return None
    if response.status_code == 200:
        return response.json()
    print(f"   Error {response.status_code}: {response.text[:120]}", flush=True)
    return None


def _search_paginated(endpoint, query, max_results=1000, *, page_delay=2.0):
    per_page = 100
    max_results = min(max_results, GITHUB_SEARCH_RESULT_CAP)
    all_items, total_count, page = [], 0, 1
    while len(all_items) < max_results:
        payload = _search_page(endpoint, query, page=page, per_page=per_page)
        if not payload:
            break
        if page == 1:
            total_count = int(payload.get("total_count", 0))
        items = payload.get("items", [])
        if not items:
            break
        all_items.extend(items[: max_results - len(all_items)])
        if len(items) < per_page or len(all_items) >= max_results:
            break
        page += 1
        if page * per_page > GITHUB_SEARCH_RESULT_CAP:
            break
        time.sleep(page_delay)
    return all_items, total_count


def _repo_record(repo, query, discovery_source, match_url=None, matched_path="", matched_file=""):
    created_at = repo.get("created_at") or repo.get("pushed_at") or repo.get("updated_at")
    age_months = 0.0
    if created_at:
        try:
            created = datetime.strptime(created_at, "%Y-%m-%dT%H:%M:%SZ")
            age_months = (datetime.utcnow() - created).days / 30
        except ValueError:
            age_months = 0.0
    repo_url = repo.get("html_url") or f"https://github.com/{repo.get('full_name', '')}"
    return {
        "full_name": repo.get("full_name"),
        "url": repo_url,
        "match_url": match_url or repo_url,
        "description": (repo.get("description") or "")[:120],
        "language": repo.get("language"),
        "stars": repo.get("stargazers_count", 0),
        "forks": repo.get("forks_count", 0),
        "open_issues": repo.get("open_issues_count", 0),
        "size_kb": repo.get("size", 0),
        "created_at": repo.get("created_at"),
        "updated_at": repo.get("updated_at"),
        "age_months": round(age_months, 1),
        "topics": ", ".join(repo.get("topics", []) or []),
        "detection_query": query,
        "discovery_source": discovery_source,
        "matched_path": matched_path,
        "matched_file": matched_file,
    }


@lru_cache(maxsize=8192)
def _fetch_repo_details(full_name):
    response = _gh_get(f"{BASE_URL}/repos/{full_name}")
    if response is not None and response.status_code == 200:
        return response.json()
    return {"full_name": full_name, "html_url": f"https://github.com/{full_name}"}


def search_repos(query, max_results=1000):
    items, total = _search_paginated("search/repositories", query, max_results=max_results)
    return [_repo_record(r, query, "repo_search") for r in items], total


def search_code(query, max_results=1000):
    if not TOKEN_POOL.has_token:
        print(f"   skipping code search for '{query}' (needs a token)", flush=True)
        return [], 0
    items, total = _search_paginated("search/code", query, max_results=max_results)
    repos = []
    for item in items:
        repository = item.get("repository")
        if not repository:
            continue
        details = _fetch_repo_details(repository["full_name"])
        repos.append(
            _repo_record(
                details, query, "code_search",
                match_url=item.get("html_url"),
                matched_path=item.get("path", ""),
                matched_file=item.get("name", ""),
            )
        )
    return repos, total

## Step 6 — SpecStory parser + enriched extraction

For each user prompt, also captures the **following agent turn**: full response, fenced **generated code**, languages, tool-use flags, and agent model name when present in the speaker marker.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

REPO_EXTRACTION_TIMEOUT_SEC = 15
MAX_SPECSTORY_FILES_PER_REPO = 30

ASSISTANT_ROLES = {"assistant", "agent", "ai"}
_SPEAKER_MODEL_RE = re.compile(r"\(([^)]+)\)")
_CODE_FENCE_RE = re.compile(r"```(\w+)?[^\n]*\n(.*?)```", re.DOTALL)

SPECSTORY_ROOT = ".specstory"
SPECSTORY_HISTORY = ".specstory/history"
SPECSTORY_PATH_FRAGMENT = ".specstory/history"

_ROLE_WORDS = r"(User|Human|Developer|Assistant|Agent|AI|System|Tool)"
SPEAKER_PATTERNS = (
    re.compile(r"^\s*#{1,6}\s*\*{0,3}\s*" + _ROLE_WORDS + r"\b.*$", re.IGNORECASE),
    re.compile(r"^\s*_{0,3}\*{1,3}\s*" + _ROLE_WORDS + r"\b.*$", re.IGNORECASE),
    re.compile(r"^\s*" + _ROLE_WORDS + r"\s*:\s*.*$", re.IGNORECASE),
)
ROLE_COLON_RE = re.compile(r"^\s*" + _ROLE_WORDS + r"\s*:\s*(.*)$", re.IGNORECASE)
USER_ROLES = {"user", "human", "developer"}
PROMPT_FIELD_RE = re.compile(r"^\s*(prompt|query|request|instruction)\s*:\s*(.+?)\s*$", re.IGNORECASE)
CJK_RE = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]")

_HTML_COMMENT_RE = re.compile(r"<!--.*?-->", re.DOTALL)
_THINK_BLOCK_RE = re.compile(r"<think\b.*?</think>", re.DOTALL | re.IGNORECASE)
_TOOL_BLOCK_RE = re.compile(r"<tool-use\b.*?</tool-use>", re.DOTALL | re.IGNORECASE)
_SEPARATOR_LINES = {"---", "***", "___"}


def _normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _speaker_role(line):
    for pattern in SPEAKER_PATTERNS:
        match = pattern.match(line)
        if match:
            return match.group(1).lower()
    return None


def _inline_after_role(line):
    match = ROLE_COLON_RE.match(line)
    return match.group(2).strip() if match else ""


def _has_cjk(text):
    return bool(CJK_RE.search(text))


def _clean_turn(lines):
    text = "\n".join(lines)
    text = _HTML_COMMENT_RE.sub("", text)
    text = _THINK_BLOCK_RE.sub("", text)
    text = _TOOL_BLOCK_RE.sub("", text)
    kept = [ln for ln in text.splitlines() if ln.strip() not in _SEPARATOR_LINES]
    return _normalize_text("\n".join(kept))


def _is_meaningful_prompt(text):
    stripped = text.strip()
    if len(stripped) < 5:
        return False
    if len(stripped.split()) >= 3:
        return True
    return _has_cjk(stripped) and len(stripped) >= 6


@dataclass
class SpecStoryPromptRecord:
    full_name: str
    repo_url: str
    source_file: str
    turn_index: int
    prompt_text: str
    agent_response: str = ""
    agent_model: str = ""
    generated_code: str = ""
    code_block_count: int = 0
    code_languages: str = ""
    has_generated_code: bool = False
    has_tool_use: bool = False


def _request_json(url, params=None):
    response = _gh_get(url, params=params)
    if response is not None and response.status_code == 200:
        return response.json()
    return None


@lru_cache(maxsize=1024)
def _fetch_repo(full_name):
    data = _request_json(f"{BASE_URL}/repos/{full_name}")
    return data if isinstance(data, dict) else None


def _is_specstory_file(path):
    lower = path.lower().replace("\\", "/")
    return SPECSTORY_PATH_FRAGMENT in lower or (
        SPECSTORY_ROOT in lower and lower.endswith(".md")
    )


def _walk_contents(full_name, path, depth=0, max_depth=5):
    url = f"{BASE_URL}/repos/{full_name}/contents/{path}".rstrip("/")
    payload = _request_json(url)
    if payload is None:
        return []
    if isinstance(payload, dict):
        return [payload] if payload.get("type") == "file" else []
    items = []
    for entry in payload:
        if entry.get("type") == "file":
            items.append(entry)
        elif entry.get("type") == "dir" and depth < max_depth:
            items.extend(_walk_contents(full_name, entry["path"], depth + 1, max_depth))
    return items


def _search_repo_specstory_files(full_name):
    if not TOKEN_POOL.has_token:
        return []
    paths = set()
    for fragment in ("path:.specstory/history", "path:.specstory extension:md"):
        payload = _request_json(
            f"{BASE_URL}/search/code",
            params={"q": f"repo:{full_name} {fragment}", "per_page": 100, "page": 1},
        )
        if not isinstance(payload, dict):
            continue
        for item in payload.get("items", []):
            path = item.get("path")
            if path and _is_specstory_file(path):
                paths.add(path)
    return sorted(paths)


def discover_specstory_files(full_name):
    files = {}
    for root in (SPECSTORY_HISTORY, SPECSTORY_ROOT):
        for entry in _walk_contents(full_name, root):
            path = entry.get("path") or ""
            if path and _is_specstory_file(path):
                files[path] = None
    for path in _search_repo_specstory_files(full_name):
        files[path] = None
    return sorted(files)


def _decode_file_content(payload):
    content = payload.get("content")
    if not content:
        return ""
    try:
        return base64.b64decode(content.replace("\n", "")).decode("utf-8", errors="replace")
    except Exception:
        return ""


def fetch_file_text(full_name, path):
    payload = _request_json(f"{BASE_URL}/repos/{full_name}/contents/{path}")
    if not isinstance(payload, dict):
        return ""
    return _normalize_text(_decode_file_content(payload))


def _speaker_model(line):
    match = _SPEAKER_MODEL_RE.search(line)
    return match.group(1).strip() if match else ""


def _extract_code_blocks(text):
    blocks, langs = [], []
    for match in _CODE_FENCE_RE.finditer(text or ""):
        lang = (match.group(1) or "").strip().lower() or "unknown"
        code = match.group(2).strip()
        if code:
            blocks.append(code)
            langs.append(lang)
    return blocks, langs


def _turn_has_tool_use(lines):
    raw = "\n".join(lines)
    return bool(_TOOL_BLOCK_RE.search(raw))


def parse_specstory_turns(text):
    """Parse user prompts paired with the next agent turn + code metadata."""
    turns_meta = []
    current_role = None
    current_lines = []
    current_model = ""
    saw_marker = False

    def flush():
        nonlocal current_role, current_lines, current_model
        if current_role is None:
            current_lines = []
            return
        turns_meta.append({
            "role": current_role,
            "model": current_model,
            "body": _clean_turn(current_lines),
            "has_tool_use": _turn_has_tool_use(current_lines),
        })
        current_role = None
        current_lines = []
        current_model = ""

    for raw_line in text.splitlines():
        role = _speaker_role(raw_line)
        if role is not None:
            saw_marker = True
            flush()
            current_role = role
            current_model = _speaker_model(raw_line)
            inline = _inline_after_role(raw_line)
            if inline:
                current_lines.append(inline)
            continue
        if current_role is not None:
            current_lines.append(raw_line)
    flush()

    pairs = []
    if not saw_marker:
        for raw_line in text.splitlines():
            match = PROMPT_FIELD_RE.match(raw_line)
            if match:
                prompt = _normalize_text(match.group(2))
                if _is_meaningful_prompt(prompt):
                    pairs.append({
                        "prompt_text": prompt,
                        "agent_response": "",
                        "agent_model": "",
                        "generated_code": "",
                        "code_block_count": 0,
                        "code_languages": "",
                        "has_generated_code": False,
                        "has_tool_use": False,
                    })
        return pairs

    for idx, turn in enumerate(turns_meta):
        if turn["role"] not in USER_ROLES:
            continue
        prompt = turn["body"]
        if not _is_meaningful_prompt(prompt):
            continue

        agent_response = ""
        agent_model = ""
        has_tool_use = False
        if idx + 1 < len(turns_meta):
            nxt = turns_meta[idx + 1]
            if nxt["role"] in ASSISTANT_ROLES:
                agent_response = nxt["body"]
                agent_model = nxt["model"]
                has_tool_use = nxt["has_tool_use"]

        blocks, langs = _extract_code_blocks(agent_response)
        pairs.append({
            "prompt_text": prompt,
            "agent_response": agent_response,
            "agent_model": agent_model,
            "generated_code": "\n\n---\n\n".join(blocks),
            "code_block_count": len(blocks),
            "code_languages": ", ".join(sorted(set(langs))) if langs else "",
            "has_generated_code": len(blocks) > 0,
            "has_tool_use": has_tool_use,
        })

    seen, cleaned = set(), []
    for pair in pairs:
        if pair["prompt_text"] in seen:
            continue
        seen.add(pair["prompt_text"])
        cleaned.append(pair)
    return cleaned


def parse_specstory_prompts(text):
    """Back-compat: user prompts only."""
    return [p["prompt_text"] for p in parse_specstory_turns(text)]


def extract_specstory_prompts_from_repo(full_name, repo_url=None):
    repo = _fetch_repo(full_name)
    if not repo:
        return []
    repo_url = repo_url or repo.get("html_url") or f"https://github.com/{full_name}"
    records = []
    paths = discover_specstory_files(full_name)[:MAX_SPECSTORY_FILES_PER_REPO]
    for path in paths:
        text = fetch_file_text(full_name, path)
        if not text:
            continue
        for turn_index, pair in enumerate(parse_specstory_turns(text), start=1):
            records.append(SpecStoryPromptRecord(
                full_name=full_name,
                repo_url=repo_url,
                source_file=path,
                turn_index=turn_index,
                prompt_text=pair["prompt_text"],
                agent_response=pair["agent_response"],
                agent_model=pair["agent_model"],
                generated_code=pair["generated_code"],
                code_block_count=pair["code_block_count"],
                code_languages=pair["code_languages"],
                has_generated_code=pair["has_generated_code"],
                has_tool_use=pair["has_tool_use"],
            ))
    return records


def extract_specstory_prompts_from_repo_timed(full_name, repo_url=None, timeout_sec=REPO_EXTRACTION_TIMEOUT_SEC):
    """Extract prompts from one repo; skip if it exceeds timeout_sec."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(extract_specstory_prompts_from_repo, full_name, repo_url)
        try:
            records = future.result(timeout=timeout_sec)
            return records, "ok"
        except FuturesTimeoutError:
            return [], "timeout"
        except Exception:
            return [], "error"


def _print_extraction_summary(df):
    if df.empty:
        print("\nNo SpecStory prompts extracted.", flush=True)
        return
    print(f"\nSpecStory prompts : {len(df)}", flush=True)
    print(f"Repos with prompts: {df['full_name'].nunique()}", flush=True)
    print(f"Source files      : {df['source_file'].nunique()}", flush=True)
    if "has_generated_code" in df.columns:
        print(f"With generated code: {int(df['has_generated_code'].sum())}", flush=True)
    if "has_tool_use" in df.columns:
        print(f"With tool use      : {int(df['has_tool_use'].sum())}", flush=True)


def _records_to_rows(records):
    rows = []
    for record in records:
        rows.append({
            "full_name": record.full_name,
            "repo_url": record.repo_url,
            "source_file": record.source_file,
            "turn_index": record.turn_index,
            "artifact_type": "session_prompt",
            "prompt_text": record.prompt_text,
            "prompt_length": len(record.prompt_text),
            "prompt_word_count": len(record.prompt_text.split()),
            "agent_response": record.agent_response,
            "agent_response_length": len(record.agent_response),
            "agent_model": record.agent_model,
            "generated_code": record.generated_code,
            "generated_code_length": len(record.generated_code),
            "code_block_count": record.code_block_count,
            "code_languages": record.code_languages,
            "has_generated_code": record.has_generated_code,
            "has_tool_use": record.has_tool_use,
            "extraction_method": "specstory_turn_pair_parser",
        })
    return rows

## Step 7 — Incremental cached mining (target 500–800 repos with prompts)

Goal: **500–800 repositories where SpecStory logs yield real user prompts** — not just candidate repos from search.

| Run | Repos per query | Typical progress |
|-----|-----------------|------------------|
| 1st | **250** | First batch of prompt-bearing repos |
| 2nd | **500** | Grow toward 500+ good repos |
| 3rd+ | **1000** | Push toward 800 (GitHub max/query) |

**Stops a run when any of these hit:**
- `TARGET_REPOS_GOAL` (800) repos with `status: ok` in cache
- `MAX_REPOS_PER_RUN` extraction attempts this run
- `MAX_RUN_WALL_SEC` wall-clock seconds

Many discovered repos have **no** SpecStory prompts — those are marked `no_prompts` and skipped on future runs. Only repos with actual extracted prompts count toward the goal.

**Cache** (`outputs/cache/`): `candidates.csv`, `prompts.csv`, `repo_status.json`, `run_state.json`

**Outputs:** `specstory_prompts_enriched_*.csv` (all prompt rows) + `specstory_repos_with_prompts_*.csv` (one row per good repo)

In [ ]:
import json

# Progressive discovery + run caps (tune as needed)
RUN_TIERS = [250, 500, 1000]
TARGET_REPOS_MIN = 500
TARGET_REPOS_GOAL = 800
MAX_REPOS_PER_RUN = 300
MAX_RUN_WALL_SEC = 25 * 60  # 25 minutes per run
REPO_EXTRACTION_TIMEOUT_SEC = 15
RESET_CACHE = False

CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_CACHE = CACHE_DIR / "candidates.csv"
PROMPTS_CACHE = CACHE_DIR / "prompts.csv"
REPO_STATUS_CACHE = CACHE_DIR / "repo_status.json"
RUN_STATE_CACHE = CACHE_DIR / "run_state.json"


def _empty_run_state():
    return {"run_count": 0, "history": []}


def _load_run_state():
    if RUN_STATE_CACHE.exists():
        with open(RUN_STATE_CACHE, encoding="utf-8") as f:
            return json.load(f)
    return _empty_run_state()


def _save_run_state(state):
    with open(RUN_STATE_CACHE, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)


def _load_repo_status():
    if REPO_STATUS_CACHE.exists():
        with open(REPO_STATUS_CACHE, encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_repo_status(status):
    with open(REPO_STATUS_CACHE, "w", encoding="utf-8") as f:
        json.dump(status, f, indent=2)


def _tier_for_run(run_count):
    """run_count is 0-based before increment; run 1 -> 50, run 2 -> 100, ..."""
    if run_count < len(RUN_TIERS):
        return RUN_TIERS[run_count]
    return RUN_TIERS[-1]


def _reset_cache():
    for path in (CANDIDATES_CACHE, PROMPTS_CACHE, REPO_STATUS_CACHE, RUN_STATE_CACHE):
        if path.exists():
            path.unlink()
    print("Cache cleared.", flush=True)


def _load_candidates_df():
    if CANDIDATES_CACHE.exists():
        return pd.read_csv(CANDIDATES_CACHE)
    return pd.DataFrame()


def _save_candidates_df(df):
    df.to_csv(CANDIDATES_CACHE, index=False, encoding="utf-8-sig")


def _load_prompts_df():
    if PROMPTS_CACHE.exists():
        return pd.read_csv(PROMPTS_CACHE)
    return pd.DataFrame()


def _save_prompts_df(df):
    df.to_csv(PROMPTS_CACHE, index=False, encoding="utf-8-sig")


def _run_discovery(max_results_per_query):
    rows = []
    for name, q in REPO_DETECTION_QUERIES:
        print(f"[repo] {name}: up to {max_results_per_query}", flush=True)
        repos, total = search_repos(q, max_results=max_results_per_query)
        print(f"   got {len(repos)} (total on github: {total})", flush=True)
        rows.extend(repos)
    for name, q in CODE_SEARCH_QUERIES:
        print(f"[code] {name}: up to {max_results_per_query}", flush=True)
        repos, total = search_code(q, max_results=max_results_per_query)
        print(f"   got {len(repos)} (total on github: {total})", flush=True)
        rows.extend(repos)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).drop_duplicates(subset="full_name").reset_index(drop=True)


def _repos_to_extract(candidates_df, repo_status):
    """Pending repos, highest-star first for more prompts per minute."""
    skip_statuses = {"ok", "no_prompts"}
    pending = []
    for row in candidates_df.itertuples(index=False):
        full_name = getattr(row, "full_name")
        status = repo_status.get(full_name, {}).get("status")
        if status in skip_statuses:
            continue
        pending.append(row)
    if pending and hasattr(pending[0], "stars"):
        pending.sort(key=lambda r: getattr(r, "stars", 0) or 0, reverse=True)
    return pending


def _count_repos_with_prompts(repo_status):
    """Repos where extraction succeeded and at least one prompt was found."""
    return sum(1 for info in repo_status.values() if info.get("status") == "ok")


def _build_repos_with_prompts_df(candidates_df, repo_status):
    ok_names = [name for name, info in repo_status.items() if info.get("status") == "ok"]
    if not ok_names or candidates_df.empty:
        return pd.DataFrame()
    repos_df = candidates_df[candidates_df["full_name"].isin(ok_names)].copy()
    repos_df["prompt_count"] = repos_df["full_name"].map(
        lambda n: repo_status.get(n, {}).get("prompt_count", 0)
    )
    repos_df["extraction_status"] = "ok"
    return repos_df.sort_values("prompt_count", ascending=False).reset_index(drop=True)


def run_incremental_specstory_mining():
    if RESET_CACHE:
        _reset_cache()

    state = _load_run_state()
    repo_status = _load_repo_status()
    cached_prompts = _load_prompts_df()
    starting_prompts = len(cached_prompts)
    starting_repos_ok = _count_repos_with_prompts(repo_status)

    if starting_repos_ok >= TARGET_REPOS_GOAL:
        print(
            f"Already at goal: {starting_repos_ok} repos with prompts "
            f"(target {TARGET_REPOS_GOAL}).",
            flush=True,
        )
        ts = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M")
        prompts_path = OUTPUT_DIR / f"specstory_prompts_enriched_{ts}.csv"
        cached_prompts.to_csv(prompts_path, index=False, encoding="utf-8-sig")
        candidates_df = _load_candidates_df()
        repos_path = OUTPUT_DIR / f"specstory_repos_with_prompts_{ts}.csv"
        _build_repos_with_prompts_df(candidates_df, repo_status).to_csv(repos_path, index=False, encoding="utf-8-sig")
        return prompts_path, candidates_df

    run_number = state["run_count"] + 1
    max_per_query = _tier_for_run(state["run_count"])
    run_start = time.time()

    print("=" * 62, flush=True)
    print(
        f"Run #{run_number}  |  {max_per_query} repos/query  |  "
        f"goal {TARGET_REPOS_MIN}-{TARGET_REPOS_GOAL} repos with prompts  |  "
        f"timeout {REPO_EXTRACTION_TIMEOUT_SEC}s/repo",
        flush=True,
    )
    print(
        f"Repos with prompts so far: {starting_repos_ok}  |  "
        f"prompt rows cached: {starting_prompts}",
        flush=True,
    )
    print("=" * 62, flush=True)

    discovered = _run_discovery(max_results_per_query=max_per_query)
    cached_candidates = _load_candidates_df()
    if cached_candidates.empty:
        candidates_df = discovered
    elif discovered.empty:
        candidates_df = cached_candidates
    else:
        candidates_df = (
            pd.concat([cached_candidates, discovered], ignore_index=True)
            .drop_duplicates(subset="full_name", keep="first")
            .reset_index(drop=True)
        )
    _save_candidates_df(candidates_df)
    new_repo_count = max(0, len(candidates_df) - len(cached_candidates))
    print(f"\nCandidates: {len(candidates_df)} total ({new_repo_count} new this run)", flush=True)

    pending = _repos_to_extract(candidates_df, repo_status)
    print(f"Repos to extract this run: {len(pending)} (cap {MAX_REPOS_PER_RUN})", flush=True)

    new_rows = []
    stats = {"ok": 0, "no_prompts": 0, "timeout": 0, "error": 0, "stopped_early": False}
    stop_reason = None

    for index, row in enumerate(pending, start=1):
        if index > MAX_REPOS_PER_RUN:
            stop_reason = f"MAX_REPOS_PER_RUN ({MAX_REPOS_PER_RUN})"
            stats["stopped_early"] = True
            break
        if time.time() - run_start > MAX_RUN_WALL_SEC:
            stop_reason = f"MAX_RUN_WALL_SEC ({MAX_RUN_WALL_SEC}s)"
            stats["stopped_early"] = True
            break
        if _count_repos_with_prompts(repo_status) >= TARGET_REPOS_GOAL:
            stop_reason = f"TARGET_REPOS_GOAL ({TARGET_REPOS_GOAL} repos with prompts)"
            stats["stopped_early"] = True
            break

        full_name = getattr(row, "full_name")
        repo_url = getattr(row, "url", None)
        t0 = time.time()
        print(f"[{index:03d}/{min(len(pending), MAX_REPOS_PER_RUN)}] {full_name}", end="", flush=True)

        records, status = extract_specstory_prompts_from_repo_timed(
            full_name,
            repo_url=repo_url,
            timeout_sec=REPO_EXTRACTION_TIMEOUT_SEC,
        )
        elapsed = time.time() - t0

        if status == "timeout":
            print(f"  -> TIMEOUT (>{REPO_EXTRACTION_TIMEOUT_SEC}s), skipping", flush=True)
            repo_status[full_name] = {
                "status": "timeout",
                "elapsed_sec": round(elapsed, 1),
                "updated_at": pd.Timestamp.utcnow().isoformat(),
            }
            stats["timeout"] += 1
            continue

        if status == "error":
            print(f"  -> ERROR ({elapsed:.1f}s), skipping", flush=True)
            repo_status[full_name] = {
                "status": "error",
                "elapsed_sec": round(elapsed, 1),
                "updated_at": pd.Timestamp.utcnow().isoformat(),
            }
            stats["error"] += 1
            continue

        if not records:
            print(f"  -> no prompts ({elapsed:.1f}s)", flush=True)
            repo_status[full_name] = {
                "status": "no_prompts",
                "prompt_count": 0,
                "elapsed_sec": round(elapsed, 1),
                "updated_at": pd.Timestamp.utcnow().isoformat(),
            }
            stats["no_prompts"] += 1
            continue

        print(f"  -> {len(records)} prompts ({elapsed:.1f}s)", flush=True)
        repo_status[full_name] = {
            "status": "ok",
            "prompt_count": len(records),
            "elapsed_sec": round(elapsed, 1),
            "updated_at": pd.Timestamp.utcnow().isoformat(),
        }
        stats["ok"] += 1
        new_rows.extend(_records_to_rows(records))

    _save_repo_status(repo_status)

    if new_rows:
        new_prompts_df = pd.DataFrame(new_rows)
        prompts_df = (
            pd.concat([cached_prompts, new_prompts_df], ignore_index=True)
            if not cached_prompts.empty
            else new_prompts_df
        )
    else:
        prompts_df = cached_prompts

    _save_prompts_df(prompts_df)

    ts = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M")
    candidate_csv = OUTPUT_DIR / f"specstory_candidates_{ts}.csv"
    prompts_path = OUTPUT_DIR / f"specstory_prompts_enriched_{ts}.csv"
    repos_path = OUTPUT_DIR / f"specstory_repos_with_prompts_{ts}.csv"
    candidates_df.to_csv(candidate_csv, index=False, encoding="utf-8-sig")
    prompts_df.to_csv(prompts_path, index=False, encoding="utf-8-sig")
    repos_with_prompts_df = _build_repos_with_prompts_df(candidates_df, repo_status)
    repos_with_prompts_df.to_csv(repos_path, index=False, encoding="utf-8-sig")

    repos_ok_total = len(repos_with_prompts_df)
    state["run_count"] += 1
    state["history"].append({
        "run_number": run_number,
        "max_per_query": max_per_query,
        "candidates_total": len(candidates_df),
        "repos_with_prompts_total": repos_ok_total,
        "repos_extracted_ok_this_run": stats["ok"],
        "repos_timeout": stats["timeout"],
        "repos_error": stats["error"],
        "repos_no_prompts": stats["no_prompts"],
        "prompts_total": len(prompts_df),
        "prompts_added": len(prompts_df) - starting_prompts,
        "stopped_early": stats["stopped_early"],
        "stop_reason": stop_reason,
        "finished_at": pd.Timestamp.utcnow().isoformat(),
    })
    _save_run_state(state)

    print("\n--- Run summary ---", flush=True)
    print(f"  Repos w/ prompts : {repos_ok_total} (+{repos_ok_total - starting_repos_ok} this run)", flush=True)
    print(f"  Extracted OK     : {stats['ok']} repos this run", flush=True)
    print(f"  No prompts       : {stats['no_prompts']}", flush=True)
    print(f"  Timeouts         : {stats['timeout']}", flush=True)
    print(f"  Errors           : {stats['error']}", flush=True)
    print(f"  Prompt rows total: {len(prompts_df)} (+{len(prompts_df) - starting_prompts} this run)", flush=True)
    print(f"  Repos CSV        : {repos_path.name}", flush=True)
    if stop_reason:
        print(f"  Stopped early    : {stop_reason}", flush=True)

    if repos_ok_total >= TARGET_REPOS_GOAL:
        print(f"\nGoal reached ({TARGET_REPOS_GOAL}+ repos with prompts). Ready for Step 9.", flush=True)
    elif repos_ok_total >= TARGET_REPOS_MIN:
        print(
            f"\nMinimum met ({TARGET_REPOS_MIN}+ repos). "
            f"Re-run to push toward {TARGET_REPOS_GOAL}.",
            flush=True,
        )
    else:
        next_tier = _tier_for_run(state["run_count"])
        print(
            f"\nRe-run Step 7 ({repos_ok_total}/{TARGET_REPOS_MIN} repos min, "
            f"{next_tier} repos/query next).",
            flush=True,
        )

    _print_extraction_summary(prompts_df)
    return prompts_path, candidates_df


prompts_path, candidates_df = run_incremental_specstory_mining()
candidates_df.head()

## Step 7b — Inspect cache (optional)

Re-run **Step 7** until **repos with prompts** reaches 500–800. Check `repo_status.json` — only `ok` repos count toward the goal.

In [ ]:
# Optional: inspect cache without re-running discovery/extraction
if RUN_STATE_CACHE.exists():
    with open(RUN_STATE_CACHE, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2))
if REPO_STATUS_CACHE.exists():
    status = _load_repo_status()
    counts = {}
    for info in status.values():
        key = info.get("status", "unknown")
        counts[key] = counts.get(key, 0) + 1
    repos_ok = counts.get("ok", 0)
    print("Repo status counts:", counts)
    print(f"Repos with prompts: {repos_ok} / goal {TARGET_REPOS_GOAL}")
if PROMPTS_CACHE.exists():
    pdf = pd.read_csv(PROMPTS_CACHE)
    print(f"Total prompt rows cached: {len(pdf)} from {pdf['full_name'].nunique()} repos")
    if "has_generated_code" in pdf.columns:
        print(f"  rows with generated code: {int(pdf['has_generated_code'].sum())}")

## Step 8b (alternative) — Upload an existing SpecStory CSV

Skip Step 7 and run this instead if you already have a `specstory_prompts_*.csv` from a local or prior run.

In [ ]:
# Run ONLY if you want to upload your own CSV instead of mining above.
from google.colab import files

uploaded = files.upload()
prompts_path = list(uploaded.keys())[0]
print("Using uploaded file:", prompts_path)

## Step 9 — Load & clean the prompt dataset

Removes noise before embedding/clustering/analysis:
- SpecStory XML command markers (`<local-command-*`, `<command-name>`)
- Slash commands, interrupted requests, IDE/agent UI action artifacts (e.g. `@agent Continue: "..."`)
- Confirmation/continuation replies that carry no independent intent (`Ok, continue.`, `Sounds good, go ahead!`, `next checklist item please`) — these can't be predicted from prompt text alone since their meaning depends entirely on the prior turn, so they're noise for the prediction task, not signal
- Pasted CI runner logs and browser console dumps
- Very short fragments and very long log pastes
- **Per-repo cap** (`max_prompts_per_repo`): a handful of chatty repos otherwise dominate the corpus (e.g. 50%+ of rows from 5 repos), which biases both training vocabulary and the repo-held-out evaluation. Repos over the cap are downsampled **stratified by `action_type`** (or by code/tool-use signal if labels aren't available yet), so a repo's mix of implement/debug/explain/review turns is preserved instead of being lost to chance in a flat random sample; repos under the cap are untouched.

If you already have cluster/UMAP columns from a prior run, they are **dropped** so Steps 10–14 recompute on the cleaned corpus.


In [ ]:
import importlib.util
import sys
from pathlib import Path

def _load_cleaning_module():
    try:
        from github_repo_miner.data_cleaning import (
            CleaningConfig,
            clean_prompts_dataframe,
            print_cleaning_report,
        )
        return CleaningConfig, clean_prompts_dataframe, print_cleaning_report
    except ImportError:
        pass

    for rel in (
        Path("src/github_repo_miner/data_cleaning.py"),
        Path("../src/github_repo_miner/data_cleaning.py"),
    ):
        if rel.exists():
            spec = importlib.util.spec_from_file_location("data_cleaning", rel)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod.CleaningConfig, mod.clean_prompts_dataframe, mod.print_cleaning_report

    raise ImportError(
        "Could not load data_cleaning. Re-run Step 1 (pip install) or run from the repo checkout."
    )

CleaningConfig, clean_prompts_dataframe, print_cleaning_report = _load_cleaning_module()

CLEANING_CONFIG = CleaningConfig(
    min_chars=25,
    min_words=4,
    min_chars_cjk=18,
    min_cjk_chars=8,
    max_chars=8_000,
    max_prompts_per_repo=50,  # skew control: no single repo can contribute more than this
)

prompts_df = pd.read_csv(prompts_path)
print("Rows loaded:", len(prompts_df))

_stale_cols = [
    "cluster", "umap_x", "umap_y",
    "char_len", "word_len", "line_count",
    "has_question", "has_code_mention", "starts_imperative",
    "intent_debug_fix", "intent_implement", "intent_refactor",
    "intent_explain", "intent_test", "intent_documentation",
    "intent_review", "intent_configure", "intent_ui_frontend",
    "intent_data_sql", "intent_count", "primary_intent",
    "agent_len", "code_to_text_ratio",
]
prompts_df = prompts_df.drop(columns=[c for c in _stale_cols if c in prompts_df.columns])

for col, default in [
    ("agent_response", ""), ("generated_code", ""), ("agent_model", ""),
    ("code_languages", ""), ("code_block_count", 0),
    ("has_generated_code", False), ("has_tool_use", False),
]:
    if col not in prompts_df.columns:
        prompts_df[col] = default

prompts_df, dropped_df, cleaning_report = clean_prompts_dataframe(
    prompts_df,
    config=CLEANING_CONFIG,
)

print("\n=== CLEANING REPORT ===")
print_cleaning_report(cleaning_report)

if not dropped_df.empty:
    print("\nTop drop reasons:")
    print(dropped_df["_drop_reason"].value_counts().head(8).to_string())
    dropped_path = OUTPUT_DIR / "specstory_prompts_dropped.csv"
    dropped_df.to_csv(dropped_path, index=False, encoding="utf-8-sig")
    print("Saved dropped rows:", dropped_path)

cleaned_path = OUTPUT_DIR / "specstory_prompts_cleaned.csv"
prompts_df.to_csv(cleaned_path, index=False, encoding="utf-8-sig")
print("Saved cleaned rows:", cleaned_path)

texts = prompts_df["prompt_text"].tolist()
print("\nUnique SpecStory prompts to embed:", len(texts))
print("Repos represented:", prompts_df["full_name"].nunique())

per_repo = prompts_df["full_name"].value_counts()
top5_share = 100 * per_repo.head(5).sum() / len(prompts_df)
top10_share = 100 * per_repo.head(10).sum() / len(prompts_df)
print(f"Top 5 repos share : {top5_share:.1f}%  (dropped {cleaning_report.dropped_repo_cap} rows to the per-repo cap)")
print(f"Top 10 repos share: {top10_share:.1f}%")
print(per_repo.head(10).to_string())
prompts_df.head()

## Step 10 — Embed prompts on GPU

Uses `all-MiniLM-L6-v2` sentence embeddings. This is the GPU-accelerated step.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("Embeddings shape:", embeddings.shape)

## Step 11 — Cluster prompts (pick k by silhouette)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

candidate_ks = [k for k in range(4, 21) if k < len(texts)]
best_k, best_score, best_labels = None, -1.0, None

for k in candidate_ks:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)
    print(f"k={k:2d}  silhouette={score:.4f}")
    if score > best_score:
        best_k, best_score, best_labels = k, score, labels

if best_labels is None:
    best_k, best_labels = 1, np.zeros(len(texts), dtype=int)

print(f"\nBest k = {best_k} (silhouette = {best_score:.4f})")
prompts_df["cluster"] = best_labels

## Step 12 — Visualize clusters (UMAP 2D)

In [ ]:
import umap
import matplotlib.pyplot as plt

n_neighbors = min(15, max(2, len(texts) - 1))
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, metric="cosine", random_state=42)
coords = reducer.fit_transform(embeddings)
prompts_df["umap_x"] = coords[:, 0]
prompts_df["umap_y"] = coords[:, 1]

plt.figure(figsize=(10, 7))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=best_labels, cmap="tab20", s=10, alpha=0.75)
plt.title(f"SpecStory prompts — {best_k} clusters (UMAP)")
plt.colorbar(sc, label="cluster")
plt.tight_layout()
plt.savefig("outputs/prompt_clusters_umap.png", dpi=150)
plt.show()

## Step 13 — Inspect clusters (sample prompts per cluster)

In [ ]:
for c in sorted(prompts_df["cluster"].unique()):
    sub = prompts_df[prompts_df["cluster"] == c]
    print(f"\n=== Cluster {c}  ({len(sub)} prompts) ===")
    for t in sub["prompt_text"].head(5):
        snippet = t.replace("\n", " ")
        print("  -", (snippet[:160] + "...") if len(snippet) > 160 else snippet)

## Step 14 — Save clustered dataset (intermediate)

Saves the clustered dataset before Part B analysis. **Step 24** saves the final annotated dataset + research report.

In [ ]:
clustered_csv = "outputs/specstory_prompts_clustered.csv"
prompts_df.to_csv(clustered_csv, index=False, encoding="utf-8-sig")
np.save("outputs/prompt_embeddings.npy", embeddings)
print("Saved:", clustered_csv, "and outputs/prompt_embeddings.npy")

from google.colab import files
files.download(clustered_csv)
files.download("outputs/prompt_clusters_umap.png")

---

# Part B — Research Analysis

Research-focused exploration of the mined SpecStory corpus. Produces tables, figures, and a written report in `outputs/analysis/`.

**Sections:** corpus overview → mining yield → prompt linguistics → developer intent → agent/code patterns → cluster profiles → diversity → final report

## Step 15 — Corpus overview & mining funnel

In [ ]:
from collections import Counter

ANALYSIS_DIR = OUTPUT_DIR / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

repo_status = _load_repo_status() if REPO_STATUS_CACHE.exists() else {}
status_counts = Counter(info.get("status", "unknown") for info in repo_status.values())
candidates_n = len(candidates_df) if "candidates_df" in dir() and candidates_df is not None else None
if candidates_n is None and CANDIDATES_CACHE.exists():
    candidates_n = len(pd.read_csv(CANDIDATES_CACHE))

repos_with_prompts = status_counts.get("ok", prompts_df["full_name"].nunique())
repos_no_prompts = status_counts.get("no_prompts", 0)
repos_timeout = status_counts.get("timeout", 0)
repos_error = status_counts.get("error", 0)

overview = pd.Series({
    "prompt_rows": len(prompts_df),
    "unique_repos": prompts_df["full_name"].nunique(),
    "unique_source_files": prompts_df["source_file"].nunique() if "source_file" in prompts_df.columns else np.nan,
    "clusters_k": prompts_df["cluster"].nunique() if "cluster" in prompts_df.columns else np.nan,
    "silhouette": round(best_score, 4) if "best_score" in dir() else np.nan,
    "candidates_discovered": candidates_n,
    "repos_with_prompts": repos_with_prompts,
    "repos_no_prompts": repos_no_prompts,
    "repos_timeout": repos_timeout,
    "repos_error": repos_error,
})
print("=== CORPUS OVERVIEW ===")
print(overview.to_string())
if candidates_n:
    print(f"\nMining yield: {repos_with_prompts / candidates_n * 100:.1f}% of discovered repos have prompts")

if status_counts:
    fig, ax = plt.subplots(figsize=(8, 4))
    labels = ["candidates", "ok", "no_prompts", "timeout", "error"]
    vals = [candidates_n or 0, repos_with_prompts, repos_no_prompts, repos_timeout, repos_error]
    ax.bar(labels, vals, color=["#4C72B0", "#55A868", "#C44E52", "#CCB974", "#8172B2"])
    ax.set_title("Mining funnel: discovered repos → extraction outcomes")
    ax.set_ylabel("repo count")
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals + [1]) * 0.01, str(v), ha="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / "01_mining_funnel.png", dpi=150)
    plt.show()


## Step 16 — Repository-level analysis

Prompt distribution across repos: concentration, stars, language, prompts-per-repo.

In [ ]:
repo_stats = (
    prompts_df.groupby("full_name")
    .agg(
        prompt_count=("prompt_text", "count"),
        files_used=("source_file", "nunique") if "source_file" in prompts_df.columns else ("prompt_text", "count"),
        with_code=("has_generated_code", "sum") if "has_generated_code" in prompts_df.columns else ("prompt_text", "count"),
    )
    .reset_index()
)

meta = candidates_df if "candidates_df" in dir() and candidates_df is not None else pd.DataFrame()
if meta.empty and CANDIDATES_CACHE.exists():
    meta = pd.read_csv(CANDIDATES_CACHE)
if not meta.empty:
    keep = [c for c in ["full_name", "stars", "language", "age_months", "forks"] if c in meta.columns]
    repo_stats = repo_stats.merge(meta[keep].drop_duplicates("full_name"), on="full_name", how="left")

repo_stats = repo_stats.sort_values("prompt_count", ascending=False)
print("Top 15 repos by prompt count:")
cols = [c for c in ["full_name", "prompt_count", "stars", "language", "with_code"] if c in repo_stats.columns]
print(repo_stats[cols].head(15).to_string(index=False))
print("\nPrompts per repo:\n", repo_stats["prompt_count"].describe().round(2))

rc = repo_stats["prompt_count"].sort_values(ascending=False).values
pareto_80 = int(np.searchsorted(np.cumsum(rc) / rc.sum(), 0.80) + 1)
print(f"Pareto: top {pareto_80} repos ({pareto_80/len(rc)*100:.1f}%) supply 80% of prompts")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(repo_stats["prompt_count"], bins=30, color="#4C72B0", edgecolor="white")
axes[0].set_title("Prompts per repo")
axes[0].set_xlabel("prompts")
if "stars" in repo_stats.columns and repo_stats["stars"].notna().any():
    axes[1].scatter(repo_stats["stars"], repo_stats["prompt_count"], alpha=0.5, s=20)
    axes[1].set_xscale("log")
    axes[1].set_title("Stars vs prompts/repo")
    axes[1].set_xlabel("stars (log)")
if "language" in repo_stats.columns:
    lt = repo_stats["language"].fillna("Unknown").value_counts().head(10)
    axes[2].barh(lt.index[::-1], lt.values[::-1], color="#55A868")
    axes[2].set_title("Top languages")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "02_repo_level.png", dpi=150)
plt.show()
repo_stats.to_csv(ANALYSIS_DIR / "repo_stats.csv", index=False, encoding="utf-8-sig")


## Step 17 — Prompt linguistics & structure

Length, questions, imperatives, multi-turn signals — what do developers actually type?

In [ ]:
prompts_df["char_len"] = prompts_df["prompt_text"].str.len()
prompts_df["word_len"] = prompts_df["prompt_text"].str.split().str.len()
prompts_df["line_count"] = prompts_df["prompt_text"].str.count("\n") + 1
prompts_df["has_question"] = prompts_df["prompt_text"].str.contains(r"\?", regex=True)
prompts_df["has_code_mention"] = prompts_df["prompt_text"].str.contains(
    r"```|\.py|\.js|\.ts|function |class |import ", case=False, regex=True
)
prompts_df["starts_imperative"] = prompts_df["prompt_text"].str.match(
    r"^(add|create|fix|update|remove|delete|implement|write|build|make|refactor|explain|help|can you|please)\b",
    case=False,
)

ling = pd.Series({
    "median_chars": prompts_df["char_len"].median(),
    "median_words": prompts_df["word_len"].median(),
    "median_lines": prompts_df["line_count"].median(),
    "pct_questions": round(prompts_df["has_question"].mean() * 100, 1),
    "pct_imperative_open": round(prompts_df["starts_imperative"].mean() * 100, 1),
    "pct_mention_code": round(prompts_df["has_code_mention"].mean() * 100, 1),
    "pct_multiline_gt3": round((prompts_df["line_count"] > 3).mean() * 100, 1),
})
print("=== PROMPT LINGUISTICS ===")
print(ling.to_string())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(prompts_df["word_len"], bins=40, color="#4C72B0", edgecolor="white")
axes[0, 0].set_title("Word count distribution")
axes[0, 1].hist(prompts_df["char_len"], bins=40, color="#55A868", edgecolor="white")
axes[0, 1].set_title("Character count distribution")
for ax, col, title in [
    (axes[1, 0], "has_question", "Contains ?"),
    (axes[1, 1], "starts_imperative", "Imperative opener"),
]:
    vc = prompts_df[col].value_counts()
    ax.bar(["no", "yes"], [vc.get(False, 0), vc.get(True, 0)], color=["#C44E52", "#55A868"])
    ax.set_title(title)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "03_prompt_linguistics.png", dpi=150)
plt.show()


## Step 18 — Developer intent taxonomy (heuristic)

Keyword-based intent labels for research segmentation. A prompt can match multiple intents.

In [ ]:
INTENT_RULES = {
    "debug_fix": r"\b(fix|bug|error|broken|crash|fail|debug|issue|exception|stack trace)\b",
    "implement": r"\b(add|create|implement|build|write|make|develop|scaffold)\b",
    "refactor": r"\b(refactor|clean up|reorganize|simplify|optimize|rewrite)\b",
    "explain": r"\b(explain|what does|how does|why does|understand|clarify|walk me through)\b",
    "test": r"\b(test|unit test|coverage|pytest|jest|spec\b|assert)\b",
    "documentation": r"\b(document|readme|docstring|comment|changelog)\b",
    "review": r"\b(review|check|look at|audit|inspect|evaluate)\b",
    "configure": r"\b(configure|setup|install|deploy|docker|ci/cd|pipeline|env)\b",
    "ui_frontend": r"\b(ui|css|style|layout|component|react|button|modal|frontend)\b",
    "data_sql": r"\b(sql|query|database|table|migration|schema|csv|dataframe)\b",
}

for intent, pattern in INTENT_RULES.items():
    prompts_df[f"intent_{intent}"] = prompts_df["prompt_text"].str.contains(pattern, case=False, regex=True)

intent_cols = [f"intent_{k}" for k in INTENT_RULES]
prompts_df["intent_count"] = prompts_df[intent_cols].sum(axis=1)
prompts_df["primary_intent"] = prompts_df[intent_cols].apply(
    lambda row: row.idxmax().replace("intent_", "") if row.any() else "unclassified", axis=1
)

intent_rates = {k: round(prompts_df[f"intent_{k}"].mean() * 100, 1) for k in INTENT_RULES}
intent_df = pd.Series(intent_rates).sort_values(ascending=False)
print("=== INTENT PREVALENCE (% of prompts matching) ===")
print(intent_df.to_string())
print(f"\nUnclassified: {(prompts_df['primary_intent'] == 'unclassified').mean()*100:.1f}%")
print(f"Multi-intent (2+ labels): {(prompts_df['intent_count'] >= 2).mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(10, 6))
intent_df.sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Heuristic developer intent prevalence")
ax.set_xlabel("% of prompts")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "04_intent_taxonomy.png", dpi=150)
plt.show()
intent_df.to_csv(ANALYSIS_DIR / "intent_prevalence.csv", encoding="utf-8-sig")


## Step 19 — Agent output & code-generation patterns

How often do agent turns include code? Which languages? Tool use?

In [ ]:
if "agent_response" in prompts_df.columns:
    prompts_df["agent_len"] = prompts_df["agent_response"].fillna("").str.len()
    prompts_df["code_to_text_ratio"] = np.where(
        prompts_df["agent_len"] > 0,
        prompts_df["generated_code"].fillna("").str.len() / prompts_df["agent_len"],
        0,
    )
else:
    prompts_df["agent_len"] = 0
    prompts_df["code_to_text_ratio"] = 0

code_stats = pd.Series({
    "prompts_with_agent_response": round((prompts_df["agent_len"] > 50).mean() * 100, 1),
    "prompts_with_generated_code": round(prompts_df["has_generated_code"].mean() * 100, 1)
    if "has_generated_code" in prompts_df.columns else np.nan,
    "prompts_with_tool_use": round(prompts_df["has_tool_use"].mean() * 100, 1)
    if "has_tool_use" in prompts_df.columns else np.nan,
    "median_code_blocks": prompts_df["code_block_count"].median()
    if "code_block_count" in prompts_df.columns else np.nan,
    "median_code_to_text_ratio": round(prompts_df["code_to_text_ratio"].median(), 3),
})
print("=== AGENT / CODE GENERATION ===")
print(code_stats.to_string())

lang_counter = Counter()
if "code_languages" in prompts_df.columns:
    for val in prompts_df["code_languages"].dropna():
        for lang in str(val).split(","):
            lang = lang.strip()
            if lang:
                lang_counter[lang] += 1
if lang_counter:
    print("\nTop generated-code languages:")
    for lang, cnt in lang_counter.most_common(15):
        print(f"  {lang}: {cnt}")

if "agent_model" in prompts_df.columns:
    models = prompts_df["agent_model"].replace("", np.nan).dropna()
    if len(models):
        print("\nAgent model markers (top 10):")
        print(models.value_counts().head(10).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
if "has_generated_code" in prompts_df.columns:
    vc = prompts_df["has_generated_code"].value_counts()
    axes[0].bar(["no code", "has code"], [vc.get(False, 0), vc.get(True, 0)], color=["#C44E52", "#55A868"])
    axes[0].set_title("Prompts where agent produced code")
if lang_counter:
    top = lang_counter.most_common(10)
    axes[1].barh([x[0] for x in top][::-1], [x[1] for x in top][::-1], color="#4C72B0")
    axes[1].set_title("Generated code languages")
axes[2].hist(
    prompts_df.loc[prompts_df["code_to_text_ratio"] > 0, "code_to_text_ratio"],
    bins=25, color="#8172B2", edgecolor="white",
)
axes[2].set_title("Code-to-response length ratio")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "05_code_generation.png", dpi=150)
plt.show()


## Step 20 — Cluster research profiles

Thematic buckets: TF-IDF terms, intent mix, and code rate per cluster.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

cluster_profiles = []
vectorizer = TfidfVectorizer(max_features=2000, stop_words="english", min_df=2)
tfidf = vectorizer.fit_transform(prompts_df["prompt_text"])
terms = np.array(vectorizer.get_feature_names_out())

for c in sorted(prompts_df["cluster"].unique()):
    mask = prompts_df["cluster"] == c
    sub = prompts_df[mask]
    mean_tfidf = np.asarray(tfidf[mask].mean(axis=0)).ravel()
    top_idx = mean_tfidf.argsort()[-8:][::-1]
    top_terms = ", ".join(terms[top_idx])
    intent_mix = {ic.replace("intent_", ""): round(sub[ic].mean() * 100, 1) for ic in intent_cols}
    dominant_intent = max(intent_mix, key=intent_mix.get) if intent_mix else "n/a"
    cluster_profiles.append({
        "cluster": c,
        "size": len(sub),
        "pct_of_corpus": round(len(sub) / len(prompts_df) * 100, 1),
        "top_terms": top_terms,
        "dominant_intent": dominant_intent,
        "pct_with_code": round(sub["has_generated_code"].mean() * 100, 1)
        if "has_generated_code" in sub.columns else np.nan,
        "median_words": sub["word_len"].median(),
    })

cluster_df = pd.DataFrame(cluster_profiles)
print("=== CLUSTER PROFILES ===")
print(cluster_df.to_string(index=False))
cluster_df.to_csv(ANALYSIS_DIR / "cluster_profiles.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(cluster_df["cluster"].astype(str), cluster_df["size"], color="#4C72B0")
ax.set_title(f"Cluster sizes (k={best_k}, silhouette={best_score:.3f})")
ax.set_xlabel("cluster")
ax.set_ylabel("prompts")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "06_cluster_sizes.png", dpi=150)
plt.show()

print("\nSample prompts per cluster:")
for c in sorted(prompts_df["cluster"].unique())[:8]:
    row = cluster_df[cluster_df["cluster"] == c].iloc[0]
    print(f"\n--- Cluster {c} (n={row['size']}, intent~{row['dominant_intent']}) ---")
    print(f"  Terms: {row['top_terms']}")
    for t in prompts_df[prompts_df["cluster"] == c]["prompt_text"].head(2):
        snip = t.replace("\n", " ")
        print("  -", (snip[:140] + "...") if len(snip) > 140 else snip)


## Step 21 — Diversity & concentration metrics

How concentrated is the corpus in a few repos/files?

In [ ]:
def gini_coefficient(values):
    arr = np.sort(np.asarray(values, dtype=float))
    if len(arr) == 0 or arr.sum() == 0:
        return 0.0
    n = len(arr)
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * arr) / (n * arr.sum())) - (n + 1) / n)

per_repo = prompts_df.groupby("full_name").size()
per_file = prompts_df.groupby("source_file").size() if "source_file" in prompts_df.columns else per_repo

diversity = pd.Series({
    "gini_repos": round(gini_coefficient(per_repo.values), 3),
    "gini_files": round(gini_coefficient(per_file.values), 3),
    "top10_repo_share_pct": round(per_repo.sort_values(ascending=False).head(10).sum() / per_repo.sum() * 100, 1),
    "top10_file_share_pct": round(per_file.sort_values(ascending=False).head(10).sum() / per_file.sum() * 100, 1),
    "herfindahl_repos": round(((per_repo / per_repo.sum()) ** 2).sum(), 4),
    "effective_n_repos": round(1 / ((per_repo / per_repo.sum()) ** 2).sum(), 1),
})
print("=== DIVERSITY / CONCENTRATION ===")
print(diversity.to_string())
print("\nGini->1 = highly concentrated. effective_n_repos = equivalent equal contributors.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(np.cumsum(per_repo.sort_values(ascending=False).values) / per_repo.sum())
axes[0].axhline(0.8, color="red", ls="--", label="80%")
axes[0].set_title("Lorenz curve — prompts across repos")
axes[0].set_xlabel("repos (ranked)")
axes[0].set_ylabel("cumulative share")
axes[0].legend()
if "primary_intent" in prompts_df.columns:
    intents = prompts_df["primary_intent"].value_counts().head(10)
    axes[1].pie(intents.values, labels=intents.index, autopct="%1.0f%%", textprops={"fontsize": 8})
    axes[1].set_title("Primary intent distribution")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "07_diversity.png", dpi=150)
plt.show()
diversity.to_csv(ANALYSIS_DIR / "diversity_metrics.csv", encoding="utf-8-sig")


## Step 22 — Prompt intent vs agent behavior

When developers ask X, does the agent return code or use tools?

In [ ]:
if "has_generated_code" in prompts_df.columns and intent_cols:
    cross = prompts_df.groupby("primary_intent").agg(
        n=("prompt_text", "count"),
        pct_code=("has_generated_code", "mean"),
        pct_tool=("has_tool_use", "mean") if "has_tool_use" in prompts_df.columns else ("prompt_text", "count"),
        median_agent_len=("agent_len", "median") if "agent_len" in prompts_df.columns else ("prompt_text", "count"),
    ).sort_values("n", ascending=False)
    cross["pct_code"] = (cross["pct_code"] * 100).round(1)
    if "has_tool_use" in prompts_df.columns:
        cross["pct_tool"] = (cross["pct_tool"] * 100).round(1)
    print("=== INTENT -> AGENT BEHAVIOR ===")
    print(cross.head(15).to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    top = cross.head(10)
    x = np.arange(len(top))
    ax.bar(x - 0.2, top["pct_code"], width=0.4, label="% with code", color="#55A868")
    if "has_tool_use" in prompts_df.columns:
        ax.bar(x + 0.2, top["pct_tool"], width=0.4, label="% with tool use", color="#4C72B0")
    ax.set_xticks(x)
    ax.set_xticklabels(top.index, rotation=45, ha="right")
    ax.set_title("Agent output patterns by developer intent")
    ax.legend()
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / "08_intent_vs_code.png", dpi=150)
    plt.show()
    cross.to_csv(ANALYSIS_DIR / "intent_vs_agent_behavior.csv", encoding="utf-8-sig")


## Step 23 — Auto-generated research report

In [ ]:
report_lines = [
    "# SpecStory Corpus — Research Report",
    "",
    f"Generated: {pd.Timestamp.utcnow().isoformat()}",
    "",
    "## 1. Corpus scale",
    f"- **{len(prompts_df):,}** unique developer prompts",
    f"- **{prompts_df['full_name'].nunique():,}** repositories with extractable SpecStory logs",
    f"- **{prompts_df['source_file'].nunique() if 'source_file' in prompts_df.columns else 'n/a'}** session log files",
    "",
    "## 2. Mining yield",
]
if candidates_n:
    report_lines.append(
        f"- {repos_with_prompts}/{candidates_n} discovered repos yielded prompts "
        f"({repos_with_prompts/candidates_n*100:.1f}% hit rate)"
    )
report_lines += [
    f"- Repos without prompts: {repos_no_prompts} | timeouts: {repos_timeout} | errors: {repos_error}",
    "",
    "## 3. Prompt characteristics",
    f"- Median length: **{prompts_df['word_len'].median():.0f} words** / {prompts_df['char_len'].median():.0f} chars",
    f"- {ling['pct_questions']}% contain questions; {ling['pct_imperative_open']}% start with imperatives",
    f"- {ling['pct_mention_code']}% mention code/syntax explicitly",
    "",
    "## 4. Developer intent (heuristic)",
]
for intent, pct in intent_df.head(5).items():
    report_lines.append(f"- **{intent}**: {pct}% of prompts")
report_lines += [
    "",
    "## 5. Agent / code-generation behavior",
    f"- {code_stats['prompts_with_generated_code']}% of prompts paired with fenced code in agent response",
    f"- {code_stats['prompts_with_tool_use']}% involve tool-use blocks",
    f"- Median code-to-response ratio: {code_stats['median_code_to_text_ratio']}",
    "",
    "## 6. Clustering",
    f"- k={best_k} clusters (silhouette={best_score:.3f})",
]
for _, row in cluster_df.head(5).iterrows():
    report_lines.append(
        f"- Cluster {row['cluster']}: n={row['size']} ({row['pct_of_corpus']}%), "
        f"intent~{row['dominant_intent']}, terms: {row['top_terms']}"
    )
report_lines += [
    "",
    "## 7. Diversity",
    f"- Gini (repos): {diversity['gini_repos']} — top 10 repos supply {diversity['top10_repo_share_pct']}% of prompts",
    f"- Effective number of contributing repos: {diversity['effective_n_repos']}",
    "",
    "## 8. Research implications",
    "- SpecStory logs capture real session-level developer-agent interaction, not curated instruction files.",
    "- Intent taxonomy spans debugging, implementation, explanation, and configuration tasks.",
    "- Code-generation rate varies by intent — useful for studying when agents produce code vs prose.",
    "- Concentration metrics show whether findings generalize or are driven by few heavy contributors.",
    "",
    "## Output files",
    "- outputs/analysis/*.png — all figures",
    "- outputs/analysis/cluster_profiles.csv",
    "- outputs/specstory_prompts_clustered.csv — full annotated dataset",
]

report_text = "\n".join(report_lines)
report_path = ANALYSIS_DIR / "RESEARCH_REPORT.md"
report_path.write_text(report_text, encoding="utf-8")
print(report_text[:3500])
print(f"\n... full report saved to {report_path}")


## Step 24 — Save & download full analysis bundle

In [ ]:
clustered_csv = OUTPUT_DIR / "specstory_prompts_clustered.csv"
prompts_df.to_csv(clustered_csv, index=False, encoding="utf-8-sig")
np.save(OUTPUT_DIR / "prompt_embeddings.npy", embeddings)

manifest = pd.Series({
    "clustered_dataset": str(clustered_csv),
    "research_report": str(ANALYSIS_DIR / "RESEARCH_REPORT.md"),
    "repo_stats": str(ANALYSIS_DIR / "repo_stats.csv"),
    "cluster_profiles": str(ANALYSIS_DIR / "cluster_profiles.csv"),
    "figures_dir": str(ANALYSIS_DIR),
})
print("=== SAVED ARTIFACTS ===")
print(manifest.to_string())

from google.colab import files
files.download(str(clustered_csv))
files.download(str(ANALYSIS_DIR / "RESEARCH_REPORT.md"))
files.download(str(ANALYSIS_DIR / "01_mining_funnel.png"))
print("\nDownloads started. Zip outputs/analysis/ for all figures.")
